# Human vs Machine Translation Detection Notebook

## Overview
This Jupyter notebook implements a machine translation pipeline using the Llama 3.1 8B Instruct model. It reads source sentences in various languages, translates them to a target language using a large language model, and saves the translations to a file.

## Workflow

### 1. Setup & Model Loading
- Imports necessary libraries (PyTorch, Hugging Face Transformers, CSV utilities)
- Loads the model and tokenizer
- Automatically detects and uses available GPU or CPU

### 2. Data Input
Two input methods are supported:
- **Text files**: Load sentences from plain text files (one sentence per line)
- **CSV files**: Load sentences from a specific column in a CSV file
- Supports chunking of large datasets for efficient processing

### 3. Translation
- Uses an instruction-tuned LLM to translate sentences between languages
- Leverages a system prompt to ensure the model acts as an expert translator
- Generates translations with controlled output length (max 512 tokens)
- Includes response cleaning to handle formatting artifacts

### 4. Output
- Saves all translations to a text file (one translation per line)
- Provides progress tracking using tqdm progress bars
- Sample translations are displayed for verification

## Key Features
- Support for multiple source languages
- Batch processing and chunking capabilities
- GPU acceleration support
- Error handling for file operations
- Clean, structured output format

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import csv

In [ ]:
# Define the model name
model_name = "meta-llama/Llama-3.1-8B-Instruct" # Set this to the desired model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    # Load the tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token 
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype="auto",
        device_map=device
    )
    print("Model and tokenizer loaded successfully.")
    print("-" * 30)
except Exception as e:
    print(f"Error loading model: {e}")
    exit()


In [3]:
def readTxtFiles(source_file):
    try:
        with open(source_file, "r", encoding="utf-8") as f:
            original_sentences = f.readlines()
            # Remove any leading/trailing whitespace from each line
            original_sentences = [line.strip() for line in original_sentences if line.strip()]
            
        print(f"Read {len(original_sentences)} sentences from file.")
        print("-" * 30)
        return original_sentences
    except FileNotFoundError:
        print("Error: file not found.")
        exit()

In [3]:
def readCsvFile(source_file, col_name="source text"):
    try:
        sentences = []

        with open(source_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            if not reader.fieldnames or col_name not in reader.fieldnames:
                raise ValueError(f"Column '{col_name}' not found in CSV.")

            for row in reader:
                sentence = row.get(col_name, "").strip()
                if sentence:
                    sentences.append(sentence)

        print(f"Read {len(sentences)} sentences from CSV.")
        print("-" * 30)
        return sentences

    except FileNotFoundError:
        raise FileNotFoundError("Error: file not found.")

In [ ]:
source_files=[] # Set this to the list of txt source files that contain the sentences to be translated

original_sentences=[]
for source in source_files:
    original_sentences.append(readTxtFiles(source))

    print(len(original_sentences[-1]))

In [ ]:
source_file = "" # Set this to the path of the input CSV file that contains the sentences to be chunked 
sentences = readCsvFile(source_file)

# Create list of lists 
chunk_size = 200
original_sentences = [
    sentences[i:i + chunk_size] 
    for i in range(0, len(sentences), chunk_size)
]

print(f"Total chunks created: {len(original_sentences)}")


In [ ]:
from tqdm import tqdm

translations = []
system_prompt = "You are an expert translator."
orginal_languages=[] # Set this to the list of original languages corresponding to each source file, e.g., ["English", "French", "German"] if you have three source files in those languages.

target_language = "" # Set this to the target language for translation, e.g., "Swahili"
sentence_number = 5000
for j,original_language in enumerate(orginal_languages):
    # start=j*sentence_number
    # start+=45000
    start=0
    for i, sentence in enumerate(tqdm(original_sentences[j][start:start+sentence_number], desc="Translating")):
        
        # Create the prompt for the model in English
        prompt = f"""Translate the text below from {original_language} to {target_language}.
        ---
        {sentence}
        ---
        Your response must contain ONLY the translated text, without any labels, prefixes, or explanations. So output only the translated version of the above text.
        """

        
        # Format the messages for the chat template
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt", padding=True).to(device)
        generated_ids = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=512,
            pad_token_id=tokenizer.eos_token_id
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        cleaned_response = response.strip().replace("'", "").replace('"', '')
        
        translations.append(cleaned_response)

print("All sentences translated successfully.")
print("-" * 30)

Translating: 100%|█████████████████████████████████████████| 100/100 [02:38<00:00,  1.58s/it]

All sentences translated successfully.
------------------------------


In [7]:
print("Translations:")
for i, translation in enumerate(translations[:5]):
    print(f"{i+1}. {translation}")
print("-" * 30)

Translations:
1. Inatupa nafasi ya kufanya kazi kwa kutegemea mizizi yangu na kuunda mchakato wangu wa kujifunza kulingana na mchakato wa kujifunza wa wasikilizaji.
2. Umbuzi wa kudhibiti wa wale wanaohitaji kutatua hili unazua kila kitu.
3. Wakati unachukuliwa kwa madawa ya matibabu kwa kawaida, mzizi wa ginger inaweza kudhibiti kwa ufanisi ugonjwa wa ukimwi.
4. Walifanikisha malengo yao na kuunda urithi wa uhuru wa kujitegemea ambao wengine wanaweza kujitambua.
5. Uchumi ulioendelea haraka pia unaweza kudhaniwa kusababisha kiasi fulani cha uharibifu wa fedha.
------------------------------


In [ ]:
output_filename =  "" # Set this to the desired output filename for the translations, e.g., "translated_sentences.txt"
try:
    with open(output_filename, "w", encoding="utf-8") as f:
        for translation in translations:
            cleaned = translation.replace("\n", " ").replace("\r", " ")
            f.write(cleaned + "\n")
    print(f"Translations successfully saved to '{output_filename}'.")
    print("-" * 30)
except Exception as e:
    print(f"Error saving translations: {e}")
    exit()

Translations successfully saved to '/home/cmua-yhy/Desktop/MT Detection/Experiments/EU DGT Translation/data/test/sw/smol_test_llamma.txt'.
------------------------------
